Nama : Ghatfani Muhammad Ilham
Kelas : 06 TPLE 002
NIM : 231011400101

                                                UTS Teknik Kompilasi

In [1]:
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass

In [2]:
import re

class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1: Perbarui regex agar mengenali simbol '^'
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env 
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current
        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))
        elif token and token.isalpha():
            if token not in self._env:
                raise ParserError(f"Semantic Error: Undefined variable '{token}'")
            self.advance()
            return Var(token)
        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node
        raise ParserError(f"Unexpected token: {token}")

    # TUGAS 2: Implementasikan fungsi power()
    # Operator pangkat bersifat right-associative (2^3^2 = 2^(3^2))
    def power(self):
        node = self.factor()
        while self._current == '^':
            op = self._current
            self.advance()
            # Recursive call untuk right-associativity
            node = BinOp(left=node, op=op, right=self.power())
        return node

    def term(self):
        # TUGAS 3: Hubungkan hierarki ke self.power()
        node = self.power()  # Ubah dari self.factor() menjadi self.power()
        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.power())  # Ubah menjadi self.power()
        return node

    def expr(self):
        node = self.term()
        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(left=node, op=op, right=self.term())
        return node

    def generate_tac(self, node):
        if isinstance(node, Num): return str(node.value)
        if isinstance(node, Var): return node.name
        
        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)
        
        self._temp_count += 1
        temp_name = f"t{self._temp_count}"
        print(f"{temp_name} = {left_val} {node.op} {right_val}")
        return temp_name

In [3]:
# Test Case 1: Basic power operation
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()
    
    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


1. Mengapa fungsi power() harus dipanggil di dalam term(), bukan sebaliknya? Jelaskan kaitannya dengan Operator Precedence.
Jawaban:
Fungsi power() harus dipanggil di dalam term() karena operator pangkat (^) memiliki precedence (tingkat prioritas) yang lebih tinggi daripada operator perkalian (*) dan pembagian (/). Dalam hierarki operator matematika:
- Pangkat (^) memiliki prioritas tertinggi
- Perkalian (*) dan pembagian (/) memiliki prioritas menengah
- Penjumlahan (+) dan pengurangan (-) memiliki prioritas terendah

Struktur parsing yang benar adalah:
expr() → term() → power() → factor()
   ↓         ↓          ↓
   +,-       *,/        ^

Jika power() dipanggil di dalam expr() atau term() memanggil factor() langsung (tanpa power()), maka operator ^ tidak akan dikenali atau akan memiliki prioritas yang salah. Dengan term() memanggil power(), maka ekspresi seperti a ^ 2 * b akan di-parse sebagai (a ^ 2) * b, bukannya a ^ (2 * b).

2. Apa yang terjadi pada fase Analisis Semantik jika variabel z digunakan dalam kode sumber tetapi tidak ada di symbol_table?
Jawaban:
Pada fase Analisis Semantik, compiler akan memeriksa apakah semua variabel yang digunakan telah dideklarasikan sebelumnya. Dalam implementasi di atas, ketika variabel z digunakan (misalnya dalam ekspresi a + z), maka:
1. Parser akan membaca token z di fungsi factor()
2. Karena z.isalpha() bernilai True, program akan masuk ke blok elif
3. Program akan memeriksa keberadaan z di self._env (symbol table)
4. Karena z tidak ditemukan, program akan melempar (raise) ParserError dengan pesan:

Semantic Error: Undefined variable 'z'

5. Eksekusi akan berhenti dan error akan ditampilkan ke pengguna.

Ini adalah contoh semantic checking yang memastikan program yang di-compile memiliki arti yang jelas dan semua referensi variabel valid.

3. Jelaskan mengapa dalam TAC, instruksi untuk a ^ 2 harus muncul sebelum instruksi untuk +.
Jawaban:
Dalam TAC (Three Address Code), instruksi untuk a ^ 2 harus muncul sebelum instruksi untuk + karena:
   1. Evaluasi Operand: TAC mengikuti prinsip bahwa operasi yang lebih dalam (nested) pada AST harus dievaluasi terlebih dahulu. a ^ 2 adalah operand dari operasi penjumlahan (+), sehingga nilainya harus dihitung sebelum penjumlahan dilakukan.

   2. Representasi AST: Untuk ekspresi a ^ 2 + b, AST-nya berbentuk:

      (+)
      /   \
   (^)    b
   /   \
   a     2
   Traversal AST secara post-order (kiri → kanan → parent) menghasilkan urutan:
   - Kunjungi node kiri (^): generate TAC untuk a ^ 2
   - Kunjungi node kanan (b): generate TAC untuk b (hanya nama variabel)
   - Kunjungi node parent (+): generate TAC yang menggunakan hasil kedua operand

   3. Data Dependency: Instruksi penjumlahan membutuhkan nilai dari a ^ 2 sebagai operand. Dalam TAC, kita tidak bisa menjumlahkan a ^ 2 langsung karena operasi pangkat bukan operasi dasar TAC. Nilai tersebut harus disimpan di temporary variable (t1) terlebih dahulu, baru kemudian digunakan.

   4. Contoh TAC yang benar:

   assembly
   t1 = a ^ 2    # Evaluasi pangkat terlebih dahulu
   t2 = t1 + b   # Baru penjumlahan
   Urutan yang salah (sebaliknya) akan menyebabkan error karena nilai a ^ 2 belum tersedia saat akan dijumlahkan.